In [1]:
import datetime
import re
import numpy as np
import pandas as pd
import time

In [13]:
START_DATE = "01-08-2025"

In [3]:
final_values_old = {
    "gdp" : {"all" : r"^GDP \(QoQ\)  \("},
    "cpi" : {"all" :  r"^CPI \(MoM\)  \(",
             "NZD" : r"^CPI \(QoQ\)  \(",
             "AUD" : r"^CPI \(QoQ\)  \(",
             "JPY" : r"^National Core CPI \(YoY\)  \("
             },
    "unemployment" : {
        "all" : r"^Unemployment Rate  \(",
        "CHF" : r"^Unemployment Rate s.a.  \("
    },
    "employment" : {
        "all" : r"^Economic Activity \(MoM",
        "MXN" : r"^Economic Activity \(MoM",
        "CHF" : r"^Employment Level  \(",
        "JPY" : r"^Jobs/applications ratio  \(",
        "GBP" : r"^Employment Change 3M/3M \(",
        "CAD" : r"^Employment Change  \(",
        "AUD" : r"^Employment Change  \(",
        "EUR" : r"^Employment Change \(QoQ\)  \(",
        "NZD" : r"^Employment Change \(QoQ\)  \(",
        "USD" : r"^Nonfarm Payrolls  \("

    },
    "mpmi" : {
        "all" : r"^Judo Bank Australia Manufacturing PMI",
        "AUD" : r"^Judo Bank Manufacturing PMI",
        "NZD" : r"^Business NZ PMI  \(",
        "CHF" : r"^procure.ch Manufacturing PMI  \(",
        "CAD" : r"^S&P Global Manufacturing PMI  \(",
        "JPY" : r"^au Jibun Bank Manufacturing PMI  \(",
        "GBP" : r"^S&P Global Manufacturing PMI  \(",
        "EUR" : r"^HCOB Eurozone Manufacturing PMI  \(",
        "MXN" : r"^S&P Global Manufacturing PMI  \(",
        "USD" : r"^S&P Global Manufacturing PMI  \(",
        
    },
    "spmi" : {
        "all" : r"^Judo Bank Australia Manufacturing PMI",
        "AUD" : r"^Judo Bank Services PMI",
        "NZD" : r"^Business NZ PMI  \(",
        "CHF" : r"^procure.ch Manufacturing PMI  \(",
        "CAD" : r"^Ivey PMI  \(",
        "JPY" : r"^au Jibun Bank Services PMI  \(",
        "GBP" : r"^S&P Global Services PMI  \(",
        "EUR" : r"^HCOB Eurozone Services PMI  \(",
        "MXN" : r"^S&P Global Manufacturing PMI  \(",
        "USD" : r"^S&P Global Services PMI  \(",
    },
    "retail" : {
        "all" : r"^Retail Sales \(MoM\)  \(",
        "USD" : r"^Retail Control \(MoM\)  \(",
        "JPY" : r"^Retail Sales \(YoY\)  \(",
        "CHF" : r"^Retail Sales \(YoY\)  \(",
        "NZD" : r"^Retail Sales \(QoQ\)  \(",

    },
    "ppi" : {
        "all" : r"^PPI \(MoM\)  \(",
        "GBP" : r"^PPI Output \(MoM\)  \(",
        "CAD" : r"^IPPI \(MoM\)  \(",
        "AUD" : r"^PPI \(QoQ\)  \(",
        "NZD" : r"^PPI Output \(QoQ\)  \(",

    },
    "interest" : {
        "all" : r"^RBA Interest Rate Decision  \(",
        "AUD" : r"^RBA Interest Rate Decision  \(",
        "MXN" : r"^Interest Rate Decision",
        "NZD" : r"^RBNZ Interest Rate Decision",
        "CHF" : r"^SNB Interest Rate Decision  \(",
        "JPY" : r"^BoJ Interest Rate Decision",
        "CAD" : r"^BoC Interest Rate Decision",
        "GBP" : r"^BoE Interest Rate Decision  \(",
        "EUR" : r"^ECB Interest Rate Decision  \(",
        "USD" : r"^Fed Interest Rate Decision",

    }
}

final_values = {
    "gdp": {
        "USD": r"^GDP Growth Rate QoQ \(Q\d\)",
        "EUR": r"^GDP Growth Rate QoQ \(",
        "GBP": r"^GDP MoM \(",
        "JPY": r"^GDP Growth Rate QoQ \(Q\d\)",
        "CAD": r"^GDP MoM \(",
        "CHF": r"^GDP Growth Rate QoQ \(Q\d\)",
        "NZD": r"^GDP Growth Rate QoQ \(Q\d\)",
        "MXN": r"^GDP Growth Rate QoQ \(Q\d\)",
        "AUD": r"^GDP Growth Rate QoQ \(Q\d\)",
    },
    "cpi": {
        "USD": r"^CPI \(",
        "EUR": r"^CPI \(",
        "GBP": r"^Inflation Rate MoM \(",
        "JPY": r"^Tokyo Core CPI YoY \(",
        "CAD": r"^Inflation Rate MoM \(",
        "CHF": r"^Inflation Rate MoM \(",
        "NZD": r"^Inflation Rate QoQ \(Q\d\)",
        "MXN": r"^Inflation Rate MoM \(",
        "AUD": r"^Inflation Rate QoQ \(",
    },
    "unemployment": {
        "USD": r"^Unemployment Rate \(",
        "EUR": r"^Unemployment Rate Harmonised \(",
        "GBP": r"^Unemployment Rate \(",
        "JPY": r"^Unemployment Rate \(",
        "CAD": r"^Unemployment Rate \(",
        "CHF": r"^Unemployment Rate \(",
        "NZD": r"^Unemployment Rate \(Q\d\)",
        "MXN": r"^Unemployment Rate \(",
        "AUD": r"^Unemployment Rate \(",
    },
    "employment": {
        "USD": r"^Nonfarm Payrolls",
        "EUR": r"^Employment Change QoQ \(",
        "GBP": r"^Employment Change \(",
        "JPY": r"^Jobs\/applications ratio \(",
        "CAD": r"^Employment Change \(",
        "CHF": r"^Non Farm Payrolls \(Q\d\)",
        "NZD": r"^Employment Change QoQ \(Q\d\)",
        "MXN": r"^Economic Activity MoM \(",
        "AUD": r"^Employment Change \(",
    },
    "mpmi": {
        "USD": r"^ISM Manufacturing PMI \(",
        "EUR": r"^HCOB Manufacturing PMI \(",
        "GBP": r"^Markit/CIPS Manufacturing PMI \(",
        "JPY": r"^Jibun Bank Manufacturing PMI \(",
        "CAD": r"^Markit Manufacturing PMI \(",
        "CHF": r"^procure\.ch Manufacturing PMI \(",
        "MXN": r"^Markit Manufacturing PMI \(",
        "NZD": r"^Business NZ PMI \(",
        "AUD": r"^Markit Manufacturing PMI \(",
    },
    "spmi": {
        "USD": r"^ISM Services PMI \(",
        "EUR": r"^HCOB Services PMI \(",
        "GBP": r"^Markit/CIPS UK Services PMI \(",
        "JPY": r"^Jibun Bank Services PMI \(",
        "CAD": r"^S&P Global Services PMI \(",
        "NZD": r"^Services NZ PSI \(",
        "AUD": r"^Markit Services PMI \(",
    },
    "retail": {
        "USD": r"^Retail Sales MoM \(",
        "EUR": r"^Retail Sales MoM \(",
        "GBP": r"^Retail Sales MoM \(",
        "JPY": r"^Retail Sales MoM \(",
        "CAD": r"^Retail Sales MoM \(",
        "CHF": r"^Retail Sales MoM \(",
        "NZD": r"^Retail Sales QoQ \(Q\d\)",
        "MXN": r"^Industrial Production MoM \(",
        "AUD": r"^Retail Sales MoM \(",
    },
    "ppi": {
        "USD": r"^PPI MoM \(",
        "EUR": r"^PPI MoM \(",
        "GBP": r"^PPI Output MoM \(",
        "JPY": r"^PPI MoM \(",
        "CAD": r"^PPI MoM \(",
        "CHF": r"^Producer & Import Prices MoM \(",
        "NZD": r"^PPI Output QoQ \(Q\d\)",
        "MXN": r"^Producer Price Index MoM \(",
        "AUD": r"^PPI QoQ \(",
    },
    "interest": {
        "USD": r"^Fed Interest Rate Decision",
        "EUR": r"^ECB Interest Rate Decision",
        "GBP": r"^BoE Interest Rate Decision",
        "JPY": r"^BoJ Interest Rate Decision",
        "CAD": r"^BoC Interest Rate Decision",
        "CHF": r"^SNB Interest Rate Decision",
        "NZD": r"^RBNZ Interest Rate Decision",
        "MXN": r"^Interest Rate Decision",
        "AUD": r"^RBA Interest Rate Decision",
    }
}

target = ['USD', 'EUR', 'GBP', 'JPY', 'CAD', 'CHF', 'NZD', 'MXN', 'AUD']

zone_mapping = {
    "EUR" : "euro zone",
    'USD' : 'united states',
    "GBP" : 'united kingdom',
    "JPY" : 'japan',
    "CAD" : 'canada',
    "CHF" : 'switzerland',
    "NZD" : 'new zealand',
    "MXN" : 'mexico',
    "AUD" : 'australia'
}

weights = {
    "gdp": 0.20,              # Broad measure of economic health
    "cpi": 0.25,              # Direct link to inflation and central bank policy decisions
    "unemployment": 0.10,     # Labor market health but less dynamic than employment change
    "employment": 0.20,       # High-frequency labor market indicator
    "mpmi": 0.07,             # Manufacturing PMI: sector-specific indicator
    "spmi": 0.08,             # Services PMI: significant in service-heavy economies
    "retail": 0.10,           # Retail Sales MoM: critical for consumer spending
    "ppi": 0.05,              # Producer Price Index: important for understanding inflation trends
    "interest": 0.05          # Interest Rate: significant for monetary policy but less frequent
}


In [5]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from datetime import datetime,timedelta
import  time
from concurrent.futures import ThreadPoolExecutor, as_completed
from random import uniform





class MyFXBookScraperParallel:
    BASE_URL = "https://widget.myfxbook.com/calendar/search.html"
    HEADERS = {
        "content-type": "application/json"
    }

    def __init__(self, start_date, end_date=None, currencies=None, max_workers=10):
        self.start_date = datetime.strptime(start_date, "%d-%m-%Y")
        self.end_date = datetime.strptime(end_date, "%d-%m-%Y") if end_date else datetime.today()
        self.currencies = currencies or ["USD"]
        self.max_workers = max_workers

    def _get_intervals(self):
        intervals = []
        current = self.start_date
        while current <= self.end_date:
            end = min(current + timedelta(days=2), self.end_date)
            intervals.append((current, end))
            current = end + timedelta(days=1)
        return intervals

    def _fetch_chunk(self, start_date, end_date):
        session = requests.Session()
        payload = {
            "startDate": start_date.strftime("%Y-%m-%dT00:00:00.000Z"),
            "endDate": end_date.strftime("%Y-%m-%dT23:59:59.999Z"),
            "language": "en",
            "impacts": ["3", "2", "1", "0"],
            "currencies": self.currencies
        }
        for attempt in range(3):
            try:
                res = session.post(self.BASE_URL, json=payload, headers=self.HEADERS, timeout=10)
                if res.status_code == 200:
                    df = self.parse_html(res.content.decode())
                    print(f"✅ {start_date.date()} to {end_date.date()} - {len(df)} events")
                    return df
                else:
                    print(f"❌ Error {res.status_code} for {start_date} to {end_date}")
                    wait = uniform(1, 30)
                    print(f"Retry {attempt+1} for {start_date} - sleeping {wait:.2f}s")
                    time.sleep(wait)
            except Exception as e:
                print(f"⚠️ Exception for {start_date} to {end_date}: {e}")
                wait = uniform(1, 5)
                print(f"Retry {attempt+1} for {start_date} - sleeping {wait:.2f}s")
                time.sleep(wait)
        return pd.DataFrame()

    def fetch_data(self):
        intervals = self._get_intervals()
        all_data = []

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {
                executor.submit(self._fetch_chunk, start, end): (start, end)
                for start, end in intervals
            }
            for future in as_completed(futures):
                df = future.result()
                if not df.empty:
                    all_data.append(df)

        return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

    @staticmethod
    def parse_html(html_content):
        soup = BeautifulSoup(html_content, 'html.parser')
        rows = soup.find_all('tr', attrs={'data-calendar-row': True})

        data = []
        for row in rows:
            try:
                date_td = row.find('td', attrs={'data-event-date': True})
                date = pd.to_datetime(int(date_td['data-event-date']), unit='ms').strftime('%Y-%m-%d %H:%M') if date_td else None

                currency_div = row.find_all('td')[2].find_all('div')[1]
                event_div = row.find_all('td')[2].find_all('div')[2]
                currency = currency_div.text.strip() if currency_div else None
                event = event_div.text.strip() if event_div else None

                impact = row.find_all('td')[3].text.strip()
                values = row.find_all('td')[4:7]
                prev = MyFXBookScraperParallel.convert_value(values[0].text.strip())
                cons = MyFXBookScraperParallel.convert_value(values[1].text.strip())
                act = MyFXBookScraperParallel.convert_value(values[2].text.strip())

                data.append({
                    'Date': date,
                    'Currency': currency,
                    'Event': event,
                    'Impact': impact,
                    'Actual': act,
                    'Consensus': cons,
                    'Previous': prev
                })
            except Exception:
                continue
        return pd.DataFrame(data)

    @staticmethod
    def convert_value(value):
        if not value or value == '-':
            return None
        value = value.replace(',', '')
        match = re.match(r'([\d\.]+)([KMB%]*)', value)
        if not match:
            return value
        num, suffix = match.groups()
        num = float(num)
        if suffix == 'K':
            num *= 1_000
        elif suffix == 'M':
            num *= 1_000_000
        elif suffix == 'B':
            num *= 1_000_000_000
        elif '%' in suffix:
            return f"{num}%"
        return int(num) if num.is_integer() else num

  

/Users/vintex/Documents/work/freelance/cot-back/workenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [18]:
def get_current_date():
  """Gets the current date in the format dd/mm/yyyy.

  Returns:
    A string representing the current date in the format dd/mm/yyyy.
  """

  today = datetime.date.today()
  return today.strftime('%d/%m/%Y')


def get_month_range(start_date, end_date):
  """Generates a list of the first day of each month between the given start and end dates.

  Args:
    start_date: A string representing the start date in the format dd/mm/yyyy.
    end_date: A string representing the end date in the format dd/mm/yyyy.

  Returns:
    A list of strings, each representing the first day of a month in the range, in the format dd/mm/yyyy.
  """

  # Convert start and end dates to datetime objects
  start_date = datetime.datetime.strptime(start_date, '%d/%m/%Y')
  end_date = datetime.datetime.strptime(end_date, '%d/%m/%Y')

  # Initialize a list to store the monthly dates
  month_range = []

  # Iterate through each month in the range
  while start_date <= end_date:
    # Add the first day of the current month to the list
    month_range.append(start_date.strftime('%d/%m/%Y'))

    # Increment the start date to the next month
    start_date = start_date + datetime.timedelta(days=31)
    start_date = start_date.replace(day=1)
  
  current_date = get_current_date()
  if current_date != month_range[-1]:
    month_range.append(current_date)

  return month_range


def combine_dataframes(dataframes):
  """Combines an array of DataFrames into a single DataFrame.

  Args:
    dataframes: An array of DataFrames to be combined.

  Returns:
    A single DataFrame containing the combined data.
  """

  if not dataframes:
    raise ValueError("Dataframes array cannot be empty.")

  # Check if all DataFrames have the same columns
  if not all(df.columns.equals(dataframes[0].columns) for df in dataframes[1:]):
    raise ValueError("DataFrames must have the same columns.")

  # Concatenate the DataFrames along the rows (axis=0)
  combined_df = pd.concat(dataframes, axis=0, ignore_index=True)

  return combined_df


# Function to extract numeric values using regex and replace None with 0
def extract_numeric(value):
  if value is None or value in ['None', 'N/A']:
    return 0  # Handle None or N/A values by replacing them with 0

  value = str(value).replace(',', '')  # Remove commas if present

  # Regex to extract numeric part, including negative sign
  match = re.search(r'(-?\d+\.?\d*)', value)
  if match:
    number = float(match.group(1))

    # Handle suffixes
    if 'K' in value:
      return number * 1000
    elif 'M' in value:
      return number * 1000000
    elif 'B' in value:
      return number * 1000000000
    elif '%' in value:
      return number / 100  # Convert percentage to decimal

    return number  # Return the extracted number as float

  return 0  # Return 0 if no number is found

def calculate_percentage_changes(df):
    """Calculate percentage change for Actual, Consensus, and Previous values."""

    def calc_percentage_change(current, previous):
        """Calculate percentage change, avoiding division by zero."""
        if previous and previous != 0:  
            return (current - previous) / abs(previous)
        return 0  

    # Ensure numeric columns have no NaN values
    df[['num_actual', 'num_forecast', 'num_previous', 'previous_previous']] = df[
        ['num_actual', 'num_forecast', 'num_previous', 'previous_previous']
    ].fillna(0)

    # Apply calculations while handling percentage values correctly
    df['actual_percentage'] = df.apply(
        lambda row: row['num_actual'] if "%" in str(row['Actual']) else calc_percentage_change(row['num_actual'], row['num_previous']), axis=1
    )

    df['forecast_percentage'] = df.apply(
        lambda row: row['num_forecast'] if "%" in str(row['Consensus']) else calc_percentage_change(row['num_forecast'], row['num_previous']), axis=1
    )

    df['previous_percentage'] = df.apply(
        lambda row: row['num_previous'] if "%" in str(row['Previous']) else calc_percentage_change(row['num_previous'], row['previous_previous']), axis=1
    )

    return df

In [16]:
def fetch_data():
    all_data = []

    for currency in ["NZD"]:
        print(f"\nFetching data for {currency}...")

        scraper = MyFXBookScraperParallel(start_date=START_DATE, currencies=[currency],max_workers=10)
        df = scraper.fetch_data()

        if df.empty:
            print(f"No data found for {currency}")
        else:
            print(f"Fetched {len(df)} records for {currency}")
            all_data.append(df)
        print("waiting to reset timer")
        time.sleep(15)

    if not all_data:
        raise ValueError("No data fetched for any currency.")

    combined = pd.concat(all_data, ignore_index=True)
    return combined

In [9]:
def filter_with_event(df,query,t):
    options = final_values[query]
    if t in options.keys():
        q = options[t]
    else:
        q = options['all']
    test_data = df[df['currency'] == t]
    test_data = test_data[test_data['importance'].isin(['low','medium','high'])]
    filtered_df = test_data[test_data['event'].str.contains(q, case=False)]
    return filtered_df

def filter_data(target_currencies, combined_df):
    """Filters the DataFrame based on predefined event patterns and currencies.

    Args:
        target_currencies (list): List of currency codes to filter.
        combined_df (pd.DataFrame): The full scraped dataset.

    Returns:
        dict: Dictionary containing filtered data per currency.
    """
    all_results = {}

    for currency in target_currencies:
        temp_data = []

        for event_category, options in final_values.items():
            q = options.get(currency, options.get('all', ''))  # Get regex pattern

            # Filter DataFrame for the specific currency
            test_data = combined_df[combined_df['Currency'] == currency]

            # Filter for impact levels
            test_data = test_data[test_data['Impact'].isin(['low', 'med', 'high'])]

            # Apply regex filter to 'Event' column
            filtered_df = test_data[test_data['Event'].str.contains(q, case=False, regex=True, na=False)]

            # Convert 'Date' to datetime format
            filtered_df['datetime'] = pd.to_datetime(filtered_df['Date'], format='%Y-%m-%d %H:%M')

            # Assign event category
            filtered_df['ev'] = event_category

            # Sort by datetime
            filtered_df = filtered_df.sort_values(by='datetime')

            # Apply numeric extraction to actual, consensus, and previous values
            filtered_df['num_actual'] = filtered_df['Actual'].apply(extract_numeric)
            filtered_df['num_forecast'] = filtered_df['Consensus'].apply(extract_numeric)
            filtered_df['num_previous'] = filtered_df['Previous'].apply(extract_numeric)

            # Shift previous values to calculate changes
            filtered_df['previous_previous'] = filtered_df['num_previous'].shift(1)

            # Calculate percentage changes
            filtered_df = calculate_percentage_changes(filtered_df)

            temp_data.append(filtered_df)

        if temp_data:
            combined_result = combine_dataframes(temp_data).sort_values(by='datetime')
            if not combined_result.empty:
                all_results[currency] = combined_result

    return all_results

In [10]:
def calculate_score_with_weights(df):
    """
    Calculate scores using weights for each indicator.

    Args:
        df (pd.DataFrame): DataFrame containing forecast, actual, and indicator columns.

    Returns:
        pd.DataFrame: DataFrame with calculated scores and rescaled scores.
    """
    # Define weights for each indicator
    

    # Ensure weights sum to 1 for consistent scoring
    

    # Calculate score based on (forecast - actual) * weight
    df['Score'] = df.apply(
        lambda row: (row['forecast_percentage'] - row['actual_percentage']) * weights.get(row['ev'], 0), axis=1
    )
    df['Surprise'] = df['actual_percentage'] - df['forecast_percentage']
    
    # Trend Component: Actual - Previous
    df['Trend'] = df['actual_percentage'] - df['previous_percentage']
    
    # Magnitude Component: |Surprise| + |Trend|
    df['Magnitude'] = np.abs(df['Surprise']) + np.abs(df['Trend'])

    # Normalize the Score to the range of -20 to 20
    min_score = df['Score'].min()
    max_score = df['Score'].max()

    # Handle edge case where all scores are the same
    if max_score - min_score == 0:
        df['Rescaled Score'] = 0  # or you can set it to np.nan
    else:
        # Map scores to -20 to 20
        df['Rescaled Score'] = -20 + ((df['Score'] - min_score) * (40)) / (max_score - min_score)

    # Round the Rescaled Score to 2 decimal places
    df['Rescaled Score'] = df['Rescaled Score'].round(2)

    # Add year and month columns for grouping
    df['year'] = df['datetime'].dt.year
    df['month'] = df['datetime'].dt.month

    # Calculate avg_score grouped by indicator and month
    new_score = df.groupby(['Event', 'month'])['Score'].mean().reset_index()
    new_score.rename(columns={'Score': 'avg_score'}, inplace=True)
    df = pd.merge(df, new_score, on=['Event', 'month'], how='left')

    # Normalize the avg_score to -20 to 20
    min_avg_score = df['avg_score'].min()
    max_avg_score = df['avg_score'].max()

    # Handle edge case where all avg_score values are the same
    if max_avg_score - min_avg_score == 0:
        df['rescaled_avg_score'] = 0  # or you can set it to np.nan
    else:
        # Map avg_score to -20 to 20
        df['rescaled_avg_score'] = -20 + ((df['avg_score'] - min_avg_score) * (40)) / (max_avg_score - min_avg_score)

    # Round the rescaled avg_score to 2 decimal places
    df['rescaled_avg_score'] = df['rescaled_avg_score'].round(2)
    df['Rescaled Trend'] = 0 
    df['Trend'] = 0

    return df

In [19]:

print("#### Fetching Data ####")
combined = fetch_data()
#combined = combined.drop_duplicates(subset='id')
print("### Filtering ###")
res = filter_data(target,combined)
analyzed_result = {}
print("### Analyzing ###")
for curr in ["NZD"]:
    curr_data = res[curr]
    sorted_data = curr_data.sort_values('datetime')
    analyzed = calculate_score_with_weights(sorted_data)
    analyzed_result[curr] = analyzed


#### Fetching Data ####

Fetching data for NZD...
✅ 2025-08-01 to 2025-08-03 - 0 events✅ 2025-08-04 to 2025-08-06 - 9 events

✅ 2025-08-16 to 2025-08-18 - 4 events
✅ 2025-08-25 to 2025-08-27 - 3 events
✅ 2025-08-22 to 2025-08-24 - 2 events
✅ 2025-08-13 to 2025-08-15 - 3 events
✅ 2025-08-10 to 2025-08-12 - 5 events
✅ 2025-08-19 to 2025-08-21 - 10 events
✅ 2025-08-07 to 2025-08-09 - 1 events
✅ 2025-08-28 to 2025-08-30 - 2 events
✅ 2025-09-03 to 2025-09-05 - 1 events
✅ 2025-08-31 to 2025-09-02 - 7 events
✅ 2025-09-06 to 2025-09-08 - 1 events
✅ 2025-09-15 to 2025-09-17 - 9 events
✅ 2025-09-21 to 2025-09-23 - 3 events
✅ 2025-09-18 to 2025-09-20 - 4 events
✅ 2025-09-12 to 2025-09-14 - 2 events
✅ 2025-09-24 to 2025-09-26 - 1 events
✅ 2025-09-27 to 2025-09-29 - 0 events
✅ 2025-09-09 to 2025-09-11 - 7 events
✅ 2025-09-30 to 2025-10-02 - 5 events
✅ 2025-10-03 to 2025-10-05 - 0 events
✅ 2025-10-06 to 2025-10-08 - 7 events
✅ 2025-10-15 to 2025-10-17 - 1 events
✅ 2025-10-09 to 2025-10-11 - 2 events

/var/folders/05/g02lgq3d5gdd1rcy7jxg_bxw0000gn/T/ipykernel_34311/3350245224.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['datetime'] = pd.to_datetime(filtered_df['Date'], format='%Y-%m-%d %H:%M')
/var/folders/05/g02lgq3d5gdd1rcy7jxg_bxw0000gn/T/ipykernel_34311/3350245224.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['ev'] = event_category
/var/folders/05/g02lgq3d5gdd1rcy7jxg_bxw0000gn/T/ipykernel_34311/3350245224.py:40: SettingWithCopyWarning: 
A value is trying to be 

### Analyzing ###


In [21]:
d = analyzed_result["NZD"]

In [24]:
d  = d[d['ev'] == "cpi"]

In [25]:
d

,Date,Currency,Event,Impact,Actual,Consensus,Previous,datetime,ev,num_actual,...,Score,Surprise,Trend,Magnitude,Rescaled Score,year,month,avg_score,rescaled_avg_score,Rescaled Trend
13,2025-10-19 21:45,NZD,Inflation Rate QoQ (Q3),med,1.0%,0.9%,0.5%,2025-10-19 21:45:00,cpi,0.01,...,-0.00025,0.001,0,0.006,2.61,2025,10,-0.00025,2.61,0
